# EDA with SQL

The separate 101-row course SQL dataset is loaded into SQLite. Queries cover all ten specified tasks. Mission success and first-stage landing success are different outcomes.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


SQL_QUERIES={'01_unique_sites': 'SELECT DISTINCT Launch_Site FROM SPACEX ORDER BY Launch_Site', '02_cca_records': "SELECT Date, Booster_Version, Launch_Site FROM SPACEX WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5", '03_nasa_payload': "SELECT SUM(PAYLOAD_MASS__KG_) AS TotalPayloadKg FROM SPACEX WHERE Customer LIKE '%NASA%'", '04_v11_average': "SELECT AVG(PAYLOAD_MASS__KG_) AS AveragePayloadKg FROM SPACEX WHERE Booster_Version='F9 v1.1'", '05_first_ground': "SELECT MIN(Date) AS FirstGroundLanding FROM SPACEX WHERE Landing_Outcome='Success (ground pad)'", '06_drone_payload': "SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEX WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_>4000 AND PAYLOAD_MASS__KG_<6000 ORDER BY Date", '07_mission_outcomes': 'SELECT Mission_Outcome,COUNT(*) AS Launches FROM SPACEX GROUP BY Mission_Outcome', '08_max_payload': 'SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEX WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEX)', '09_2015_failures': "SELECT Date,Booster_Version,Launch_Site FROM SPACEX WHERE Date LIKE '2015%' AND Landing_Outcome='Failure (drone ship)'", '10_rank_outcomes': "SELECT Landing_Outcome,COUNT(*) AS Launches FROM SPACEX WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Launches DESC"}
def sql_analysis():
    df=pd.read_csv(DATA/'Spacex.csv'); df['Date']=pd.to_datetime(df.Date).dt.strftime('%Y-%m-%d')
    for col in df.select_dtypes(include=['object','str']).columns: df[col]=df[col].str.strip()
    with sqlite3.connect(RESULTS/'spacex.sqlite') as conn:
        df.to_sql('SPACEX',conn,if_exists='replace',index=False)
        outputs={}
        for name,query in SQL_QUERIES.items():
            result=pd.read_sql_query(query,conn); result.to_csv(RESULTS/(name+'.csv'),index=False)
            outputs[name]=result.to_dict('records'); print(name, '\n',query,'\n',result.to_string(index=False),'\n')
    (ROOT/'queries.sql').write_text('\n\n'.join('-- '+k+'\n'+v+';' for k,v in SQL_QUERIES.items()))
    return outputs


In [2]:
outputs=sql_analysis()

01_unique_sites 
 SELECT DISTINCT Launch_Site FROM SPACEX ORDER BY Launch_Site 
  Launch_Site
 CCAFS LC-40
CCAFS SLC-40
  KSC LC-39A
 VAFB SLC-4E 

02_cca_records 
 SELECT Date, Booster_Version, Launch_Site FROM SPACEX WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5 
       Date Booster_Version Launch_Site
2010-06-04  F9 v1.0  B0003 CCAFS LC-40
2010-12-08  F9 v1.0  B0004 CCAFS LC-40
2012-05-22  F9 v1.0  B0005 CCAFS LC-40
2012-10-08  F9 v1.0  B0006 CCAFS LC-40
2013-03-01  F9 v1.0  B0007 CCAFS LC-40 

03_nasa_payload 
 SELECT SUM(PAYLOAD_MASS__KG_) AS TotalPayloadKg FROM SPACEX WHERE Customer LIKE '%NASA%' 
  TotalPayloadKg
         107010 

04_v11_average 
 SELECT AVG(PAYLOAD_MASS__KG_) AS AveragePayloadKg FROM SPACEX WHERE Booster_Version='F9 v1.1' 
  AveragePayloadKg
           2928.4 

05_first_ground 
 SELECT MIN(Date) AS FirstGroundLanding FROM SPACEX WHERE Landing_Outcome='Success (ground pad)' 
 FirstGroundLanding
        2015-12-22 

06_drone_payload 
 SELECT Date,Booster_Ve